In [ ]:
#####################################
# This creates figure
# TraxAceBar.png 
# in paper
#####################################
import sys
# import modules in other directories
sys.path.append('../Utils/')

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature



import trax_util as trx
import ibtracs as IBT
import parse_tempest_file as TM

import importlib
import PlotUtil as Pu

importlib.reload( Pu )
importlib.reload( trx )
importlib.reload( IBT )
importlib.reload( TM )


In [ ]:
#f=trx.rcp85fname(ens=1,sst='sst3')
#f=trx.pdfname(ens=1)

ibx_basins = IBT.basin_outlines(Mediterranean=False,xNEPacific=False)

power_wind=(1./6.)**(0.11)
c5_1_gfdl =trx.readtrx( trx.pdfname(ens=1) , power_wind=power_wind )

ftrx_c5 = '/glade/work/stepheba/post/tempestextremes/cam5/trajectories.txt.f.e13.FAMIPC5.ne120_ne120.1979_2012.001'
power_wind=(1./6.)**(0.11)
c5_1_tmpst =TM.readtrx( ftrx_c5 , power_wind=power_wind, columns = ['cell_id','lon','lat','slp','wind','xxx1','year','month','day','hour',])
c5_1_tmpst_basins = trx.find_basin(c5_1_tmpst, ibx_basins)
c5_1_tmpst.basin=c5_1_tmpst_basins

ftrx_c7 ='/glade/work/stepheba/post/tempestextremes/amip/trajectories_all.txt'
power_wind=(1./2.)**(0.11)
c7_1 =TM.readtrx( ftrx_c7 , power_wind=power_wind )
c7_1_basins = trx.find_basin(c7_1,ibx_basins)
c7_1.basin=c7_1_basins



ibx=IBT.readtrx()




In [ ]:

print(c5_1_gfdl.keys())
print(c5_1_tmpst.keys())
print(c5_1_gfdl.hour.shape)
print(c5_1_tmpst.hour.shape)
print(c5_1_gfdl.hour[10,0:10])
print(c5_1_tmpst.hour[10,0:10])

#c5_1=c5_1_gfdl

In [ ]:
c5_1=c5_1_gfdl


yr0,yr1=c7_1.startyear, c7_1.endyear

ibx=trx.select_years(ibx,range=[yr0,yr1] )
c5_1=trx.select_years(c5_1,range=[yr0,yr1] )
c5_1_tmpst=trx.select_years(c5_1_tmpst,range=[yr0,yr1] )
    

In [ ]:
print(c5_1.hour.shape)
print(c5_1_tmpst.hour.shape)


In [ ]:
plt.plot( c5_1.day[100,:] )

In [ ]:
importlib.reload(trx)
#time_gfdl = trx.make_time(c5_1.year, c5_1.month, c5_1.day, c5_1.hour)
#time_tpsx = trx.make_time(c5_1_tmpst.yearB, c5_1_tmpst.month, c5_1_tmpst.day, c5_1_tmpst.hour)
timeA = trx.make_time(c5_1.year, c5_1.month, c5_1.day, c5_1.hour)
timeB = trx.make_time(c5_1_tmpst.year, c5_1_tmpst.month, c5_1_tmpst.day, c5_1_tmpst.hour)
latA,lonA = c5_1.lat,c5_1.lon
latB,lonB = c5_1_tmpst.lat,c5_1_tmpst.lon

matches = []

max_dist_km = 50.0       # tune this
min_common_times = 3      # tune this
min_close_frac = 0.5      # tune this

for ia in range(latA.shape[0]):
    tA = timeA[ia]
    goodA = ~np.isnat(tA)

    for ib in range(latB.shape[0]):
        tB = timeB[ib]
        goodB = ~np.isnat(tB)

        common_times = np.intersect1d(tA[goodA], tB[goodB])

        if len(common_times) < min_common_times:
            continue

        # indices of common times
        iax = np.array([np.where(tA == t)[0][0] for t in common_times])
        ibx = np.array([np.where(tB == t)[0][0] for t in common_times])

        dist = trx.haversine_km(
            latA[ia, iax], lonA[ia, iax],
            latB[ib, ibx], lonB[ib, ibx]
        )

        close = dist < max_dist_km

        if close.mean() >= min_close_frac:
            matches.append(dict(
                stormA=ia,
                stormB=ib,
                n_common=len(common_times),
                close_frac=close.mean(),
                mean_dist_km=dist.mean(),
                median_dist_km=np.median(dist),
                max_dist_km=dist.max()
            ))

print(len(matches))

In [ ]:
imatch=200
ia,ib = matches[imatch]['stormA'] , matches[imatch]['stormB'] 


plt.scatter(lonA[ia,:],latA[ia,:] )
plt.scatter(lonB[ib,:],latB[ib,:] )


In [ ]:
print( c5_1.startyear, c5_1.endyear, c5_1.sourcefile )
print( c5_1_tmpst.startyear, c5_1_tmpst.endyear, c5_1_tmpst.sourcefile )
print( c7_1.startyear, c7_1.endyear, c7_1.sourcefile )
print( ibx.startyear, ibx.endyear, ibx.sourcefile )


In [ ]:
x=np.linspace(0,360,num=2*3600+1 )
y=np.linspace(-90,90,num=2*1800+1 )

x=np.linspace(0,360,num=360//2+1 )
y=np.linspace(-90,90,num=180//2+1 )

lons = 0.5*( x[1:]+x[0:-1] )
lats = 0.5*( y[1:]+y[0:-1] )

Catty='Cat1+'  #'TS+'
Catty='TS+'

adens_c5_1 = trx.adensity(trx=c5_1,x=x,y=y,alpha=2, delta_t_trax=3.0, auto_scale=True  ) #,category='Cat1+')
dens_c5_1 = trx.density(trx=c5_1,x=x,y=y,category=Catty , delta_t_trax=3.0  )
print('done c5 #1')

adens_c5_1_tmpst = trx.adensity(trx=c5_1_tmpst,x=x,y=y,alpha=2, delta_t_trax=6.0, auto_scale=True  ) #,category='Cat1+')
dens_c5_1_tmpst = trx.density(trx=c5_1_tmpst,x=x,y=y,category=Catty , delta_t_trax=6.0  )
print('done c5 TEMPEST #1')

adens_c7_1 = trx.adensity(trx=c7_1,x=x,y=y,alpha=2, delta_t_trax=6.0, auto_scale=True  ) #,category='Cat1+')
dens_c7_1 = trx.density(trx=c7_1,x=x,y=y,category=Catty, delta_t_trax=6.0 )
print('done c7 #1')

Iadens = trx.adensity(trx=ibx,x=x,y=y,alpha=2, delta_t_trax=3.0, auto_scale=True ) #,category='Cat1+')
Idens = trx.density(trx=ibx,x=x,y=y,category=Catty , delta_t_trax=3.0 )

print(np.shape(x),np.shape(lons))
print(np.shape(y),np.shape(lats))

In [ ]:
# Power wind law was already applied ... so set to 1 here
power_wind=1.0 # (1./6.)**(0.11)

bace_c5_1=trx.basinACE(wind=c5_1.wind,basin=c5_1.basin,power_wind=power_wind , delta_t_trax=3. , year=c5_1.year)
#bace_c5_1=bace_c5_1 / 33.

bace_c5_1_tmpst=trx.basinACE(wind=c5_1_tmpst.wind,basin=c5_1_tmpst.basin,power_wind=power_wind , delta_t_trax=6. , year=c5_1.year)
#bace_c5_1=bace_c5_1 / 33.

bace_c7_1=trx.basinACE(wind=c7_1.wind,basin=c7_1.basin,power_wind=power_wind,  delta_t_trax=6.  , year=c7_1.year )
#bace_c7_1=bace_c7_1 / 10. #8.

Ibace=trx.basinACE(wind=ibx.wind,basin=ibx.basin,  delta_t_trax=3.,  year=ibx.year )
#Ibace=Ibace / 43.




In [ ]:
print( np.nanmin(c5_1.basin) )
print( np.min(c5_1.basin) )
print( np.nanmin(c7_1.basin) )
print( np.min(c7_1.basin) )
print( np.nanmin(ibx.basin) )
print( np.min(ibx.basin) )



In [ ]:
#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=False)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""

Axes = np.asarray( [ 
    [ .05, .6, .4, .25 ] ,
    [ .55, .6, .4, .25] ,
    [ .05, .3, .4, .25 ] ,
    [ .55, .3, .4, .25 ] ,
    [ .3, 0., .4, .25] 
    ])

print(np.shape(Axes))

print(Axes[0,:])

scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=3*np.linspace(0.2,4.0,num=20)


fig = plt.figure(figsize=(20, 15))
levs=[29,26,20,10]

p=0
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*adens_c5_1 ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM5 {c5_1.startyear}-{c5_1.endyear}" , fontsize=16)

###
# Plot IBTrACS basins
###
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    loos='-'
    if ((basi[b].number == 1) or (basi[b].number == 7)) :
        loos='--'
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs,linestyle=loos)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 

ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')
 
p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,  scale_fac*adens_c5_1_tmpst ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM5 TempestExt {c5_1_tmpst.startyear}-{c5_1_tmpst.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,  scale_fac*adens_c7_1 ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM7 {c7_1.startyear}-{c7_1.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*Iadens ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"IBTrACS {ibx.startyear}-{ibx.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')





#####################################
# ACE bars
#####################################

BarLabel = "ACE year$^{-1}$ (2$^{\circ}$)$^{-2}$"
cax = plt.axes([0.98, 0.35, 0.02, 0.4])
cbar = plt.colorbar(po,cax=cax,orientation='vertical')
cbar.set_label( label=BarLabel,size=16)
cbar.ax.tick_params(labelsize=12)



ax1 = fig.add_axes( Axes[4,:]  )

# Get some pastel shades for the bar colors
colors = plt.cm.BuPu(np.linspace(0.2, 0.5, 4))

#boo=np.arange(8)
boo=np.arange(8)
bnames=['None','N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#bnames=['N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#plt.bar( boo,np.sum(bace,axis=1),tick_label=bnames,label=bnames)
ax1.bar( boo-.4,bace_c5_1_tmpst,width=0.2,color=colors[3],label='CAM5 #1 TMPST')
ax1.bar( boo-.2,bace_c5_1,width=0.2,color=colors[0],label='CAM5 #1')
ax1.bar( boo-.0,bace_c7_1,width=0.2,color=colors[1],label='CAM7 #1' , tick_label=bnames)
#ax1.bar( boo   ,bace_c7_1,width=0.2,color=colors[2] , tick_label=bnames,label='CAM #3')

ipan=ipan+1
ax1.text(-0.15, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

#plt.bar( boo+.2,Ibace,tick_label=bnames,label=bnames,width=0.2,color='orange')
ax1.bar( boo+.2,Ibace,width=0.2,color='red',label='IBTrACS')
plt.legend()
plt.xlim(.4,8)
plt.ylabel("1980-2012 Accumulated Cyclone Energy",size=16)


Pu.save_link(png_filename = "TraxAceBar-CAM7.png")


In [ ]:
#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=False)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""

Axes = np.asarray( [ 
    [ .05, .6, .4, .25 ] ,
    [ .55, .6, .4, .25] ,
    [ .3, .3, .4, .25 ] ,
    [ .55, .3, .4, .25 ] ,
    [ .3, 0., .4, .25] 
    ])

print(np.shape(Axes))

print(Axes[0,:])

scale_fac = 1e-4 * 1.944**2 
denslv=21 #np.linspace(0,15,num=16)
#denslv=np.linspace(500,10_500,num=21)
denslv=3*np.linspace(0.2,4.0,num=20)


fig = plt.figure(figsize=(20, 15))
levs=[29,26,20,10]
    
ax1 = fig.add_axes( Axes[0,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*adens_c5_1 ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM5 {c5_1.startyear}-{c5_1.endyear}" , fontsize=16)

###
# Plot IBTrACS basins
###
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    loos='-'
    if ((basi[b].number == 1) or (basi[b].number == 7)) :
        loos='--'
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs,linestyle=loos)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 

ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')
 

ax1 = fig.add_axes( Axes[1,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,  scale_fac*adens_c7_1 ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM7 {c7_1.startyear}-{c7_1.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

ax1 = fig.add_axes( Axes[2,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*Iadens ,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"IBTrACS {ibx.startyear}-{ibx.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

BarLabel = "ACE year$^{-1}$ (2$^{\circ}$)$^{-2}$"
cax = plt.axes([0.98, 0.35, 0.02, 0.4])
cbar = plt.colorbar(po,cax=cax,orientation='vertical')
cbar.set_label( label=BarLabel,size=16)
cbar.ax.tick_params(labelsize=12)


In [ ]:
#####################################
# ACE bars
#####################################

fig = plt.figure(figsize=(10, 8))


Axes =[ .2,.2,.8,.8 ]
ax1 = fig.add_axes( Axes  )

# Get some pastel shades for the bar colors
colors = plt.cm.BuPu(np.linspace(0.2, 0.5, 3))

#boo=np.arange(8)
boo=np.arange(8)
bnames=['None','N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#bnames=['N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#plt.bar( boo,np.sum(bace,axis=1),tick_label=bnames,label=bnames)
ax1.bar( boo-.2,bace_c5_1,width=0.2,color=colors[0],label='CAM5 ')
ax1.bar( boo-.0,bace_c7_1,width=0.2,color=colors[1],label='CAM7 ' , tick_label=bnames)
#ax1.bar( boo   ,bace_c7_1,width=0.2,color=colors[2] , tick_label=bnames,label='CAM #3')

ipan=ipan+1
#ax1.text(-0.15, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
#    fontsize=20, fontweight='bold', va='top')

#plt.bar( boo+.2,Ibace,tick_label=bnames,label=bnames,width=0.2,color='orange')
ax1.bar( boo+.2,Ibace,width=0.2,color='red',label='IBTrACS')
plt.legend()
plt.xlim(.4,8)
plt.ylabel("1990-2010 Accumulated Cyclone Energy",size=16)


##Pu.save_link(png_filename = "TraxAceBar-CAM7.png")

In [ ]:
#projPC = ccrs.Robinson(central_longitude=180.)
projPC = ccrs.PlateCarree(central_longitude=180.)
data_crs = ccrs.PlateCarree()

basi = IBT.basin_outlines(Mediterranean=False,xNEPacific=False)
ipan=-1
"""
Add axes method 
ax = fig.add_axes([xmin , ymin , dx , dy , ])
"""

Axes = np.asarray( [ 
    [ .05, .6, .4, .25 ] ,
    [ .55, .6, .4, .25] ,
    [ .05, .3, .4, .25 ] ,
    [ .55, .3, .4, .25 ] ,
    [ .3, 0., .4, .25] 
    ])

print(np.shape(Axes))

print(Axes[0,:])


scale_fac = 1. #1e-4 * 1.944**2 
denslv= 200.*np.linspace(0.002,0.052,num=21) # - 0.05


fig = plt.figure(figsize=(20, 15))
levs=[29,26,20,10]

p=0
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*dens_c5_1,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM5 {c5_1.startyear}-{c5_1.endyear}" , fontsize=16)

###
# Plot IBTrACS basins
###
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    loos='-'
    if ((basi[b].number == 1) or (basi[b].number == 7)) :
        loos='--'
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs,linestyle=loos)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 

ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')
 
p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*dens_c5_1_tmpst,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM5 TempestExt {c5_1_tmpst.startyear}-{c5_1_tmpst.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*dens_c7_1,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"CAM7 {c7_1.startyear}-{c7_1.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

p=p+1
ax1 = fig.add_axes( Axes[p,:] , projection=projPC)
ax1.set_global()
ax1.coastlines(resolution='110m',color='black')
po=ax1.contourf(lons,lats,scale_fac*Idens,transform=data_crs,levels=denslv,cmap='gist_ncar')
ax1.set_title( f"IBTrACS {ibx.startyear}-{ibx.endyear}" , fontsize=16)
ipan=ipan+1
ax1.text(-0.1, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

BarLabel = "Hrs(Cat1+) year$^{-1}$ (2$^{\circ}$)$^{-2}$"
BarLabel = f"Hrs({Catty}) year{r'$^{-1}$ (2$^{\circ}$)$^{-2}$'}"
cax = plt.axes([0.98, 0.35, 0.02, 0.4])
cbar = plt.colorbar(po,cax=cax,orientation='vertical')
cbar.set_label( label=BarLabel,size=16)
cbar.ax.tick_params(labelsize=12)



"""
###
# Plot IBTrACS basins
###
bbox=dict(facecolor='white', edgecolor='black')
for b in np.arange( start=1, stop=len(basi) ):
    xyb=np.array( basi[b].xy )
    ax1.plot( xyb[:,0],xyb[:,1] ,color='red',transform=data_crs)
    if (basi[b].labloc[0]>-9999):
        ax1.text( basi[b].labloc[0],basi[b].labloc[1],basi[b].name , transform=data_crs, color='red' , bbox=bbox ) 



#####################################
# ACE bars
#####################################

BarLabel = "Hrs(Cat1+) year$^{-1}$ (2$^{\circ}$)$^{-2}$"
cax = plt.axes([0.98, 0.35, 0.02, 0.4])
cbar = plt.colorbar(po,cax=cax,orientation='vertical')
cbar.set_label( label=BarLabel,size=16)
cbar.ax.tick_params(labelsize=12)



ax1 = fig.add_axes( Axes[4,:]  )

# Get some pastel shades for the bar colors
colors = plt.cm.BuPu(np.linspace(0.2, 0.5, 3))

#boo=np.arange(8)
boo=np.arange(8)
bnames=['None','N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#bnames=['N \nAtl','S \n Atl' , 'NW \n Pac', 'NE \n Pac', 'SW \n Pac','N \n Indian','S \nIndian']
#plt.bar( boo,np.sum(bace,axis=1),tick_label=bnames,label=bnames)
ax1.bar( boo-.4,bace1,width=0.2,color=colors[0],label='CAM #1')
ax1.bar( boo-.2,bace2,width=0.2,color=colors[1],label='CAM #2')
ax1.bar( boo   ,bace3,width=0.2,color=colors[2] , tick_label=bnames,label='CAM #3')

ipan=ipan+1
ax1.text(-0.15, 1.1, f"{chr(97 +ipan)})", transform=ax1.transAxes,
    fontsize=20, fontweight='bold', va='top')

#plt.bar( boo+.2,Ibace,tick_label=bnames,label=bnames,width=0.2,color='orange')
ax1.bar( boo+.2,Ibace,width=0.2,color='red',label='IBTrACS')
plt.legend()
plt.xlim(.4,8)
plt.ylabel("1980-2012 Accumulated Cyclone Energy",size=16)

Pu.save_link(png_filename = "TraxCat1HrsBar.png")
"""